# 16 — dim_tempo

## Purpose
Generate the `dim_tempo` date dimension table covering 2021-01-01 to 2030-12-31
at day granularity. This table is a mandatory component of the Gold dimensional
model and enables clean, readable analytical queries without inline date functions.

## Input
- No external source — dates are generated programmatically via DuckDB `generate_series`.

## Output
- `local/data/gold/dim_tempo.parquet`

## Steps
1. Imports and configuration
2. Generate date spine
3. Compute calendar attributes
4. Write to Parquet
5. Validation

## Notes
- Idempotent — safe to re-run (overwrites the output file).
- No partition — ~3,652 rows; partitioning would be a small files antipattern.
- Month names and day names in English — international portfolio standard.
- fiscal_year = calendar year (no alternative fiscal calendar in this project).

## Step 1 — Imports and configuration

In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timezone
import duckdb

# --- Resolve project root via repo markers (sem depender de utils ainda) ---
def _find_root(start: Path) -> Path:
    current = start if start.is_dir() else start.parent
    for candidate in [current, *current.parents]:
        if (candidate / "utils" / "paths.py").exists() and (candidate / "notebooks").is_dir():
            return candidate
    return start

try:
    _seed = Path(__vsc_ipynb_file__).resolve()
except NameError:
    _seed = Path.cwd().resolve()

PROJECT_ROOT = _find_root(_seed)

# --- Injeta PROJECT_ROOT no sys.path para que utils/ seja encontrado ---
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Agora o import funciona ---
from utils.paths import gold_path, ensure_dir

OUTPUT_PATH = Path(gold_path(PROJECT_ROOT, "dim_tempo"))
ensure_dir(OUTPUT_PATH.parent)

DATE_START = "2021-01-01"
DATE_END   = "2030-12-31"
LOADED_AT  = datetime.now(timezone.utc).isoformat()

print(f"Project root: {PROJECT_ROOT}")
print(f"Output      : {OUTPUT_PATH}")
print(f"Range       : {DATE_START} → {DATE_END}")
print(f"Loaded      : {LOADED_AT}")

## Step 2 — Generate date spine and compute calendar attributes

In [ ]:
con = duckdb.connect()

dim_tempo = con.execute(f"""
    WITH spine AS (
        SELECT UNNEST(
            generate_series(DATE '{DATE_START}', DATE '{DATE_END}', INTERVAL '1 day')
        )::DATE AS full_date
    )
    SELECT
        -- surrogate key: YYYYMMDD as integer
        CAST(strftime(full_date, '%Y%m%d') AS INTEGER)      AS date_id,

        full_date,

        -- year / quarter
        YEAR(full_date)                                      AS year,
        QUARTER(full_date)                                   AS quarter,
        'Q' || CAST(QUARTER(full_date) AS VARCHAR)           AS quarter_label,

        -- month
        MONTH(full_date)                                     AS month,
        strftime(full_date, '%B')                            AS month_name,
        strftime(full_date, '%Y-%m')                         AS month_label,

        -- week / day
        ISODOW(full_date)                                    AS day_of_week,   -- 1=Mon, 7=Sun
        dayname(full_date)                                   AS day_name,
        WEEK(full_date)                                      AS week_of_year,

        -- weekend flags
        ISODOW(full_date) >= 6                               AS is_weekend,
        ISODOW(full_date) < 6                                AS is_weekday,

        -- fiscal year (equals calendar year for this project)
        YEAR(full_date)                                      AS fiscal_year,

        -- audit
        '{LOADED_AT}'::TIMESTAMP WITH TIME ZONE             AS _loaded_at

    FROM spine
    ORDER BY full_date
""").df()

print(f"Rows generated: {len(dim_tempo):,}")
print(dim_tempo.head(3).to_string(index=False))

## Step 3 — Write to Parquet

In [ ]:
con.execute(f"""
    COPY (
        SELECT * FROM dim_tempo
    ) TO '{OUTPUT_PATH}' (FORMAT PARQUET)
""")

file_size_mb = OUTPUT_PATH.stat().st_size / 1_048_576
print(f"Written : {OUTPUT_PATH}")
print(f"Size    : {file_size_mb:.2f} MB")

## Step 4 — Validation

In [ ]:
checks = []

result = con.execute(f"SELECT * FROM read_parquet('{OUTPUT_PATH}')").df()

# CHECK 1 — row count
expected_rows = (
    con.execute(f"""
        SELECT DATEDIFF('day', DATE '{DATE_START}', DATE '{DATE_END}') + 1
    """).fetchone()[0]
)
actual_rows = len(result)
checks.append(("Row count", actual_rows == expected_rows, f"{actual_rows} (expected {expected_rows})"))

# CHECK 2 — date_id uniqueness
unique_ids = result["date_id"].nunique()
checks.append(("date_id unique", unique_ids == actual_rows, f"{unique_ids} unique / {actual_rows} total"))

# CHECK 3 — no nulls in key columns
null_count = result[["date_id", "full_date", "year", "month"]].isnull().sum().sum()
checks.append(("No nulls in key cols", null_count == 0, f"{null_count} nulls found"))

# CHECK 4 — quarter_label distinct values
quarter_labels = sorted(result["quarter_label"].unique().tolist())
checks.append(("Quarter labels", quarter_labels == ["Q1", "Q2", "Q3", "Q4"], str(quarter_labels)))

# CHECK 5 — month_name distinct values
month_count = result["month_name"].nunique()
checks.append(("Month names (12)", month_count == 12, f"{month_count} distinct values"))

# CHECK 6 — weekend spot check (2021-01-02 is Saturday)
sat = result[result["full_date"] == "2021-01-02"]["is_weekend"].values[0]
checks.append(("is_weekend 2021-01-02 (Sat)", bool(sat) is True, str(sat)))

# CHECK 7 — weekday spot check (2021-01-04 is Monday)
mon = result[result["full_date"] == "2021-01-04"]["is_weekend"].values[0]
checks.append(("is_weekend 2021-01-04 (Mon)", bool(mon) is False, str(mon)))

# Report
print(f"\n{'CHECK':<40} {'STATUS':<8} DETAIL")
print("-" * 75)
all_pass = True
for name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    print(f"{name:<40} [{status}]   {detail}")
    if not passed:
        all_pass = False

print()
if all_pass:
    print("All checks PASSED — dim_tempo ready.")
else:
    raise AssertionError("One or more validation checks FAILED — review output above.")